# Week 10 Problem Set: The Endorsement

You are the political director from the case. Before the board meeting, you (1) check whether the vendor's turnout model can be trusted, and (2) write your recommendation.

**Lying with data — the checklist so far:**
1. **W1:** Conflating fixed and marginal costs.
2. **W2:** Presenting an observational comparison as a causal effect.
3. **W3:** Applying a result from one setting to a different one.
4. **W4:** Cherry-picking the winning arm from a multi-arm test.
5. **W5:** Treating an underpowered null as evidence of no effect.
6. **W7:** Reporting the complier comparison as a causal effect.
7. **W8:** Cherry-picking polls; house effects; ignoring nonresponse bias.
8. **W9:** Comparing by effect size without cost; ignoring uncertainty in cost-per-vote.
9. **W10:** Trusting a forecast or a model's probability **without its calibration track record**.

### Before you start

**Save your own copy first.** Go to **File → Save a copy in Drive**. A new tab opens with your own copy — work in that tab. Edits you make to the original are not saved.

**The data loads itself.** There is nothing to download or upload. The setup cell below pulls the data straight from the course repository; you just need to be online.

Stuck? See the Colab troubleshooting guide on the syllabus.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

df = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk10_forecasts_and_calibration/data/turnout_history.csv')

## Task 1: Fit the turnout model and read it (pre-filled)

Run the cell. This is the same model from livecode — a regression that predicts 2014 turnout from past voting and demographics.

In [ ]:
train = df.sample(frac=0.7, random_state=10)
test  = df.drop(train.index).copy()
model = smf.ols('voted14 ~ voted12 + voted10 + voted08 + age + female', data=train).fit()
print(model.params.round(4))

**Question 1:** The coefficient on `voted12` is about **+0.34**. In one sentence, say what that means in plain English — and why it makes "past voting predicts future voting" the foundation of any turnout model.

*Your answer:*

## Task 2: Build the calibration table yourself

The split and predictions are pre-filled. **You** build the calibration table — the same `pd.cut` + `groupby` you saw in livecode.

In [ ]:
test['pred'] = model.predict(test)
test['bin'] = pd.cut(test['pred'].clip(0, 1), bins=np.linspace(0, 1, 11))

# YOUR CODE HERE: group `test` by 'bin' and, for each bin, compute the mean predicted
# probability and the actual mean of voted14. (Recall the livecode .agg(...). Use
# observed=True.) Assign the result to `calibration`.
calibration =  # YOUR CODE HERE

print(calibration.round(3))

*Check: in the bin where the model predicts about 0.57, the actual fraction who voted should be about 0.55 — close to the prediction.*

**Question 2:** Looking at your table, is the model well-calibrated on these held-out voters? Answer in one sentence, citing one row.

**Before you move on:** Suppose the model had predicted 0.9 for a group of voters but only 0.5 of them actually voted. Where would that group's dot sit relative to the 45-degree line, and would you call the model *overconfident* or *underconfident* there? (1–2 sentences.)

*Your answers:*

## Task 3: Apply the model to a different electorate

`other_electorate.csv` is a different set of voters (a higher-turnout electorate). We apply the **same** model — no refitting — and check whether its probabilities still hold up.

In [ ]:
other = pd.read_csv('https://raw.githubusercontent.com/joshuakalla/data_science_campaigns_26/main/weeks/wk10_forecasts_and_calibration/data/other_electorate.csv').copy()
other['pred'] = model.predict(other)
print('this electorate turnout:', round(other['voted14'].mean(), 3))
print('model mean prediction:  ', round(other['pred'].mean(), 3))

# YOUR CODE HERE: compute the Brier score on this electorate -- the mean of
# (predicted - actual)^2. Clip the predictions to [0, 1] first, as in livecode.
brier_other =  # YOUR CODE HERE
print('Brier score here:', round(brier_other, 4))

*Check: the Brier score should be much worse here (\~0.22) than on the original held-out voters (\~0.14), and the model's mean prediction (\~0.47) falls well below the actual turnout (\~0.57) — it under-predicts.*

**Before you move on:** The model's coefficients never changed, yet it's wrong here. It's tempting to say "this electorate just votes more" — but is that the whole story? (Think: if these were the *same kind* of voters, just more of the high-turnout ones, the model would still be right about *which* of them votes.) In 1–2 sentences, say what this implies about trusting a vendor's "validated" model on an electorate it wasn't built for.

*Your answer:*

## Task 4: The memo — a forced advocacy (250–350 words + a final paragraph)

This memo is different. Instead of "the headline is wrong / here's the honest number / steelman," you will write the **entire memo as the strongest possible case for one position**, and reveal what you actually think only at the very end.

**Write a memo to your board arguing FOR endorsing Candidate B** — the *lower*-probability candidate (35% to win the primary, vs. the front-runner's 60%). Make the best, most honest case you can. Your memo must use all three:

**(a) The general-election argument.** B is the stronger general-election candidate (55% vs. 45% if nominated). Explain why winning the *seat* — not the nomination — is what your group should optimize, and do the arithmetic on each candidate's overall chance of holding the seat (multiply the two probabilities). **Note the multiplication actually favors A (about 0.27 vs. 0.19) — your job in the memo is to argue why that gap is smaller and less trustworthy than it looks**, using (b) and (c).

**(b) Calibration / uncertainty.** The forecaster's numbers are five months out. Using this week's language, say what you'd actually need to see to trust "60%" and "35%," and why that uncertainty helps B's case.

**(c) Heterogeneous effects.** Your endorsement plus a field program moves some candidates' primaries more than others. Argue why it would move B's primary more than A's.

Then, in a **separate final paragraph clearly labeled "What I actually believe,"** say whether you'd really make this recommendation, and why. You are allowed to argue yourself out of your own memo.

**Style rules:**
- The memo body is single-minded advocacy for B. Save your real view for the labeled last paragraph.
- 250–350 words for the body (the final paragraph is extra).
- Use at least one specific number.

**Memo to:** Advocacy group board
**From:** Political Director
**Re:** Primary endorsement recommendation

*Replace this text with your memo, then your "What I actually believe" paragraph.*

---

## Before you submit

1. **Runtime → Restart session and run all.** Do this *after* you have finished every task and written your memo. It clears every variable and runs the notebook from top to bottom, in order, so the version you hand in is one that actually works start to finish.
2. **Check that every cell actually ran.** Scroll from the top. Every code cell should show a number in its left margin and its output below it. If the run stopped at a cell with an error, that is a cell you have not finished — fix it, then restart and run all again.
3. **File → Print → Save as PDF.**
4. **Open the PDF and read it before you upload.** The PDF will look complete even when it isn't — every heading and prompt prints whether or not the code ran. What matters is the **output**: under each code cell you should see a table, a number, or a plot. A red error box, or `In [ ]` with nothing beneath it, means that part did not run and will be graded as missing. Also check that your memo printed in full and that no plot is cut off at a page break.
5. Upload the PDF to Canvas.